# Coleta de Dados — lcn-ceara

> **Requer Linux (WSL)** — rode este notebook no WSL com o ambiente virtual ativado.

Gera os seguintes arquivos em `../dados/`:
- `dados_sih_2024.csv` — todas as internações do Ceará 2024
- `dados_sih_2024_pulmao.csv` — filtrado por CID C34 (câncer de pulmão) com tipo de procedimento
- `cnes_porte_CE_2024.csv` — hospitais com classificação por porte (CONASS 2014)

## 0. Imports e configurações

In [ ]:
import os
import pandas as pd
from pysus.ftp.databases.sih import SIH
from pysus.ftp.databases.cnes import CNES
from pysus.ftp import FTPSingleton

UF    = 'CE'
ANO   = [2024]
MESES = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]

DIR_DBC   = '../dados/DBC'
DIR_DADOS = '../dados'

os.makedirs(DIR_DBC,   exist_ok=True)
os.makedirs(DIR_DADOS, exist_ok=True)

FTPSingleton.timeout = 180

print('Configurações carregadas!')

## 1. Download SIH — todas as internações CE 2024

In [ ]:
print('Carregando arquivos do SIH...')
sih = SIH().load()
print('Arquivos do SIH carregados!')

files = sih.get_files('RD', uf=UF, year=ANO, month=MESES)
print(f'{len(files)} arquivos encontrados.')

print('Baixando arquivos do SIH...')
arquivos = sih.download(files, local_dir=DIR_DBC)
print('Download concluído!')

## 2. Salvar CSV completo

In [ ]:
csv_sih = os.path.join(DIR_DADOS, 'dados_sih_2024.csv')
if os.path.exists(csv_sih):
    os.remove(csv_sih)

first = True
for arquivo in arquivos:
    df = arquivo.to_dataframe()
    df.to_csv(csv_sih, mode='a', header=first, index=False)
    first = False
    print(f'  salvo: {arquivo}')

print(f'\nCSV gerado: {csv_sih}')

## 3. Filtro câncer de pulmão (CID C34) + classificação de procedimento

In [ ]:
def classificar_procedimento(proc):
    proc = str(proc).zfill(10)
    if proc.startswith('0304'):
        return 'Tratamento Clinico'
    elif proc.startswith('03'):
        return 'Intercorrencia'
    elif proc.startswith('04'):
        return 'Cirurgia'
    else:
        return 'Outro'

csv_pulmao = os.path.join(DIR_DADOS, 'dados_sih_2024_pulmao.csv')
if os.path.exists(csv_pulmao):
    os.remove(csv_pulmao)

first = True
for arquivo in arquivos:
    df = arquivo.to_dataframe()
    df_c34 = df[df['DIAG_PRINC'].astype(str).str.startswith('C34', na=False)].copy()
    if not df_c34.empty:
        df_c34['TIPO_PROC'] = df_c34['PROC_REA'].astype(str).apply(classificar_procedimento)
        df_c34.to_csv(csv_pulmao, mode='a', header=first, index=False)
        first = False
        print(f'  {len(df_c34)} registros C34 em {arquivo}')

print(f'\nCSV gerado: {csv_pulmao}')

# Resumo
df_resumo = pd.read_csv(csv_pulmao, low_memory=False)
print('\nDistribuição por tipo de procedimento:')
print(df_resumo['TIPO_PROC'].value_counts())

## 4. Coleta CNES — classificação de hospitais por porte

In [ ]:
# Tentar carregar leitos do arquivo LT
parquet_lt = os.path.join(DIR_DBC, 'LTCE2412.parquet')

if not os.path.exists(parquet_lt):
    print('Baixando arquivo LT do CNES...')
    cnes = CNES().load()
    files_lt = cnes.get_files('LT', uf=UF, year=2024, month=12)
    arquivos_lt = cnes.download(files_lt, local_dir=DIR_DBC)
    print('Download concluído!')

df_lt = pd.read_parquet(parquet_lt)
df_lt['QT_SUS'] = pd.to_numeric(df_lt['QT_SUS'], errors='coerce').fillna(0)

df_porte = df_lt.groupby('CNES')['QT_SUS'].sum().reset_index()
df_porte.columns = ['CNES', 'TOTAL_LEITOS_SUS']

def classificar_porte(leitos):
    if leitos <= 50:
        return 'Pequeno'
    elif leitos <= 150:
        return 'Medio'
    elif leitos <= 500:
        return 'Grande'
    else:
        return 'Especial'

df_porte['CNES']  = df_porte['CNES'].astype(str)
df_porte['PORTE'] = df_porte['TOTAL_LEITOS_SUS'].apply(classificar_porte)

print('Distribuição por porte:')
print(df_porte['PORTE'].value_counts())

csv_porte = os.path.join(DIR_DADOS, 'cnes_porte_CE_2024.csv')
df_porte.to_csv(csv_porte, index=False)
print(f'\nCSV gerado: {csv_porte}')

## 5. Resumo final

In [ ]:
print('=' * 50)
print('Coleta finalizada! Arquivos gerados:')
for f in [csv_sih, csv_pulmao, csv_porte]:
    tamanho = os.path.getsize(f) / (1024 * 1024)
    print(f'  {f} ({tamanho:.1f} MB)')
print('=' * 50)